# Letter Recognition — Ensemble Classification

Multi-class classification of 26 capital letters (A–Z) from 16 pixel-statistic features. The pipeline trains individual models (Random Forest, MLP, SVM), combines them with voting ensembles, and builds a stacking meta-learner to improve accuracy.

**Dataset:** 20,000 distorted letter images from 20 fonts, each summarized by 16 integer attributes (0–15). Train on the first 16,000 samples; evaluate on the remaining 4,000.

**Features:** bounding-box position/size, on-pixel counts, statistical moments (x-bar, y-bar, variances, correlations), and edge counts for each letter image.

In [ ]:
import sys
assert sys.version_info >= (3, 5)
import sklearn
assert sklearn.__version__ >= "0.20"
import numpy as np
import os


## 1. Data Preprocessing

Load the CSV, assign column names, encode letter labels, and prepare feature matrices.

In [ ]:
import pandas as pd
def load_data(path, name):
    csv_path = os.path.join(path, name)
    return pd.read_csv(csv_path, header=None)


In [ ]:
letters = load_data(".", "letter-recognition.data.csv")


In [ ]:
letters.head()


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,T,2,8,3,5,1,8,13,0,6,6,10,8,0,8,0,8
1,I,5,12,3,7,2,10,5,5,4,13,3,9,2,8,4,10
2,D,4,11,6,8,6,10,6,2,6,10,3,7,3,7,3,9
3,N,7,11,6,6,3,5,9,4,6,4,4,10,6,10,2,8
4,G,2,1,3,1,1,8,6,6,6,6,5,9,1,7,5,10


In [ ]:
letters.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 17 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       20000 non-null  object
 1   1       20000 non-null  int64 
 2   2       20000 non-null  int64 
 3   3       20000 non-null  int64 
 4   4       20000 non-null  int64 
 5   5       20000 non-null  int64 
 6   6       20000 non-null  int64 
 7   7       20000 non-null  int64 
 8   8       20000 non-null  int64 
 9   9       20000 non-null  int64 
 10  10      20000 non-null  int64 
 11  11      20000 non-null  int64 
 12  12      20000 non-null  int64 
 13  13      20000 non-null  int64 
 14  14      20000 non-null  int64 
 15  15      20000 non-null  int64 
 16  16      20000 non-null  int64 
dtypes: int64(16), object(1)
memory usage: 2.6+ MB


In [ ]:
col_names = ['letter', 'x-box', 'y-box', 'width', 'height', 'onpix', 'x-bar', 'y-bar',
             'x2bar', 'y2bar', 'xybar', 'x2ybr', 'xy2br', 'x-ege', 'xegvy', 'y-ege', 'yegvx']


In [ ]:
print(col_names)
print(len(col_names))


['letter', 'x-box', 'y-box', 'width', 'height', 'onpix', 'x-bar', 'y-bar', 'x2bar', 'y2bar', 'xybar', 'x2ybr', 'xy2br', 'x-ege', 'xegvy', 'y-ege', 'yegvx']
17


In [ ]:
letters.columns = col_names


In [ ]:
letters.head()


,letter,x-box,y-box,width,height,onpix,x-bar,y-bar,x2bar,y2bar,xybar,x2ybr,xy2br,x-ege,xegvy,y-ege,yegvx
0,T,2,8,3,5,1,8,13,0,6,6,10,8,0,8,0,8
1,I,5,12,3,7,2,10,5,5,4,13,3,9,2,8,4,10
2,D,4,11,6,8,6,10,6,2,6,10,3,7,3,7,3,9
3,N,7,11,6,6,3,5,9,4,6,4,4,10,6,10,2,8
4,G,2,1,3,1,1,8,6,6,6,6,5,9,1,7,5,10


In [ ]:
letters['letter']


0        T
1        I
2        D
3        N
4        G
        ..
19995    D
19996    C
19997    T
19998    S
19999    A
Name: letter, Length: 20000, dtype: object

Map each letter (A–Z) to an integer label 0–25 using ASCII offset.

In [ ]:
a = np.zeros(len(letters))
for i in range(len(letters)):
    a[i] = ord(letters['letter'][i]) - ord('A')


In [ ]:
a


array([19.,  8.,  3., ..., 19., 18.,  0.], shape=(20000,))

Store encoded labels in a new target column `y`.

In [ ]:
letters['y'] = a


In [ ]:
letters.head()


,letter,x-box,y-box,width,height,onpix,x-bar,y-bar,x2bar,y2bar,xybar,x2ybr,xy2br,x-ege,xegvy,y-ege,yegvx,y
0,T,2,8,3,5,1,8,13,0,6,6,10,8,0,8,0,8,19.0
1,I,5,12,3,7,2,10,5,5,4,13,3,9,2,8,4,10,8.0
2,D,4,11,6,8,6,10,6,2,6,10,3,7,3,7,3,9,3.0
3,N,7,11,6,6,3,5,9,4,6,4,4,10,6,10,2,8,13.0
4,G,2,1,3,1,1,8,6,6,6,6,5,9,1,7,5,10,6.0


Remove the raw `letter` column — the model uses numeric features only.

In [ ]:
letters_new = letters.drop('letter', axis=1)


## 2. Train / Test Split

Use the first 16,000 rows for training and the last 4,000 for testing (dataset convention).

In [ ]:
letters.head()


,letter,x-box,y-box,width,height,onpix,x-bar,y-bar,x2bar,y2bar,xybar,x2ybr,xy2br,x-ege,xegvy,y-ege,yegvx,y
0,T,2,8,3,5,1,8,13,0,6,6,10,8,0,8,0,8,19.0
1,I,5,12,3,7,2,10,5,5,4,13,3,9,2,8,4,10,8.0
2,D,4,11,6,8,6,10,6,2,6,10,3,7,3,7,3,9,3.0
3,N,7,11,6,6,3,5,9,4,6,4,4,10,6,10,2,8,13.0
4,G,2,1,3,1,1,8,6,6,6,6,5,9,1,7,5,10,6.0


In [ ]:
letters_new.head()


,x-box,y-box,width,height,onpix,x-bar,y-bar,x2bar,y2bar,xybar,x2ybr,xy2br,x-ege,xegvy,y-ege,yegvx,y
0,2,8,3,5,1,8,13,0,6,6,10,8,0,8,0,8,19.0
1,5,12,3,7,2,10,5,5,4,13,3,9,2,8,4,10,8.0
2,4,11,6,8,6,10,6,2,6,10,3,7,3,7,3,9,3.0
3,7,11,6,6,3,5,9,4,6,4,4,10,6,10,2,8,13.0
4,2,1,3,1,1,8,6,6,6,6,5,9,1,7,5,10,6.0


In [ ]:
X_train = letters_new.iloc[0:16000,:16]
X_train.head()


,x-box,y-box,width,height,onpix,x-bar,y-bar,x2bar,y2bar,xybar,x2ybr,xy2br,x-ege,xegvy,y-ege,yegvx
0,2,8,3,5,1,8,13,0,6,6,10,8,0,8,0,8
1,5,12,3,7,2,10,5,5,4,13,3,9,2,8,4,10
2,4,11,6,8,6,10,6,2,6,10,3,7,3,7,3,9
3,7,11,6,6,3,5,9,4,6,4,4,10,6,10,2,8
4,2,1,3,1,1,8,6,6,6,6,5,9,1,7,5,10


In [ ]:
X_train.shape


(16000, 16)

In [ ]:
y_train = letters_new.iloc[0:16000,16]
y_train.head()


0    19.0
1     8.0
2     3.0
3    13.0
4     6.0
Name: y, dtype: float64

In [ ]:
y_train.shape


(16000,)

In [ ]:
X_test = letters_new.iloc[16000:,:16]
X_test.head()


,x-box,y-box,width,height,onpix,x-bar,y-bar,x2bar,y2bar,xybar,x2ybr,xy2br,x-ege,xegvy,y-ege,yegvx
16000,4,10,6,7,9,9,6,4,3,6,7,7,9,8,5,6
16001,6,9,8,4,3,8,7,3,4,13,5,8,6,8,0,8
16002,6,9,8,8,10,7,7,5,4,7,6,8,7,9,7,10
16003,5,6,6,4,3,7,6,2,7,7,6,9,0,9,4,8
16004,5,9,7,6,4,9,7,3,5,10,4,6,5,8,1,7


In [ ]:
X_test.shape


(4000, 16)

In [ ]:
y_test = letters_new.iloc[16000:,16]
y_test.head()


16000    20.0
16001    13.0
16002    21.0
16003     8.0
16004    13.0
Name: y, dtype: float64

In [ ]:
y_test.shape


(4000,)

## 3. Individual Classifiers

Train three baseline models on standardized features, then compare with 3-fold cross-validation.

### 3.1 Random Forest

Standardize training features with zero mean and unit variance.

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.astype(np.float64))


Train a random forest with 100 estimators (`random_state=42`) and evaluate with 3-fold CV.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)


,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [ ]:
from sklearn.model_selection import cross_val_score
print(cross_val_score(rf, X_train_scaled, y_train, cv=3, scoring="accuracy")  )


[0.95331834 0.95030939 0.95405963]


### 3.2 Multi-Layer Perceptron (MLP)

Train a feedforward neural network classifier and evaluate with 3-fold CV.

In [ ]:
from sklearn.neural_network import MLPClassifier
mlp = MLPClassifier(random_state=42)
mlp.fit(X_train_scaled, y_train)


/Users/mikaeldaluz/Documents/CSE4502/code/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,hidden_layer_sizes,"(100,)"
,activation,'relu'
,solver,'adam'
,alpha,0.0001
,batch_size,'auto'
,learning_rate,'constant'
,learning_rate_init,0.001
,power_t,0.5
,max_iter,200
,shuffle,True
,random_state,42


In [ ]:
print(cross_val_score(mlp, X_train_scaled, y_train, cv=3, scoring="accuracy")  )


/Users/mikaeldaluz/Documents/CSE4502/code/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/mikaeldaluz/Documents/CSE4502/code/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


[0.94131984 0.94130883 0.93905869]


/Users/mikaeldaluz/Documents/CSE4502/code/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


### 3.3 SVM (One-vs-Rest)

Wrap a kernel SVM in `OneVsRestClassifier` for 26-way classification.

In [ ]:
from sklearn.svm import SVC
from sklearn.multiclass import OneVsRestClassifier
ovr_clf = OneVsRestClassifier(SVC(gamma="auto", random_state=42, probability=True))


In [ ]:
cross_val_score(ovr_clf, X_train_scaled, y_train, cv=3, scoring="accuracy")


array([0.9135733 , 0.91449466, 0.91318207])

## 4. Hard Voting Ensemble

Combine RF, MLP, and SVM predictions by majority vote.

In [ ]:
from sklearn.ensemble import VotingClassifier
voting_clfh = VotingClassifier(
    estimators=[('rf', rf), ('mlp', mlp), ('svc', ovr_clf)],
    voting='hard'
)
voting_clfh.fit(X_train_scaled, y_train)


/Users/mikaeldaluz/Documents/CSE4502/code/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


,estimators,"[('rf', ...), ('mlp', ...), ...]"
,voting,'hard'
,weights,None
,n_jobs,None
,flatten_transform,True
,verbose,False
,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1


In [ ]:
cross_val_score(voting_clfh, X_train, y_train, cv=3, scoring="accuracy")


/Users/mikaeldaluz/Documents/CSE4502/code/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/mikaeldaluz/Documents/CSE4502/code/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/mikaeldaluz/Documents/CSE4502/code/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


array([0.9640045 , 0.95968498, 0.95949747])

## 5. Soft Voting Ensemble

Combine base models using predicted class probabilities instead of hard labels.

In [ ]:
from sklearn.ensemble import VotingClassifier
voting_clfs = VotingClassifier(
    estimators=[('rf', rf), ('mlp', mlp), ('svc', ovr_clf)],
    voting='soft'
)
voting_clfs.fit(X_train_scaled, y_train)


/Users/mikaeldaluz/Documents/CSE4502/code/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


,estimators,"[('rf', ...), ('mlp', ...), ...]"
,voting,'soft'
,weights,None
,n_jobs,None
,flatten_transform,True
,verbose,False
,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1


In [ ]:
cross_val_score(voting_clfs, X_train, y_train, cv=3, scoring="accuracy")


/Users/mikaeldaluz/Documents/CSE4502/code/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/mikaeldaluz/Documents/CSE4502/code/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/mikaeldaluz/Documents/CSE4502/code/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


array([0.96550431, 0.95968498, 0.95968498])

Evaluate the soft-voting ensemble on the held-out test set.

In [ ]:
X_test_scaled = scaler.transform(X_test.astype(np.float64))
from sklearn.metrics import accuracy_score
y_pred = voting_clfs.predict(X_test_scaled)
accuracy_score(y_test, y_pred)


0.964

## 6. Feature Importance

Rank the 16 pixel-statistic features by random forest impurity-based importance.

In [ ]:
importances = rf.feature_importances_
feature_importance = sorted(
    zip(X_train.columns, importances),
    key=lambda x: x[1],
    reverse=True
)
for feature, importance in feature_importance:
    print(f"{feature}: {importance:.4f}")


x-ege: 0.1152
y-ege: 0.1002
y2bar: 0.0930
xy2br: 0.0850
x2ybr: 0.0845
x2bar: 0.0819
xegvy: 0.0765
xybar: 0.0739
y-bar: 0.0677
yegvx: 0.0540
x-bar: 0.0531
onpix: 0.0263
y-box: 0.0252
x-box: 0.0220
width: 0.0215
height: 0.0201


## 7. Stacking Ensemble

Split the original training set into a smaller train split (12k) and validation split (4k). Base models predict on validation data; a meta-learner (blender) learns from those predictions.

Partition training data: 12,000 samples for base-model training, 4,000 for generating stacked meta-features.

In [ ]:
X_train = letters_new.iloc[0:12000,:16]
X_train.head()


,x-box,y-box,width,height,onpix,x-bar,y-bar,x2bar,y2bar,xybar,x2ybr,xy2br,x-ege,xegvy,y-ege,yegvx
0,2,8,3,5,1,8,13,0,6,6,10,8,0,8,0,8
1,5,12,3,7,2,10,5,5,4,13,3,9,2,8,4,10
2,4,11,6,8,6,10,6,2,6,10,3,7,3,7,3,9
3,7,11,6,6,3,5,9,4,6,4,4,10,6,10,2,8
4,2,1,3,1,1,8,6,6,6,6,5,9,1,7,5,10


In [ ]:
y_train = letters_new.iloc[0:12000,16]
y_train.head()


0    19.0
1     8.0
2     3.0
3    13.0
4     6.0
Name: y, dtype: float64

In [ ]:
X_val = letters_new.iloc[12000:16000,:16]
X_val.head()


,x-box,y-box,width,height,onpix,x-bar,y-bar,x2bar,y2bar,xybar,x2ybr,xy2br,x-ege,xegvy,y-ege,yegvx
12000,4,7,4,5,2,3,10,3,6,11,12,7,2,11,2,6
12001,5,9,5,7,4,4,8,5,7,11,9,14,2,9,3,7
12002,3,6,4,4,2,10,2,2,3,8,2,8,2,6,2,8
12003,5,8,7,6,6,10,6,3,6,10,4,7,4,7,5,10
12004,3,6,4,4,4,8,5,10,0,6,8,8,6,5,0,8


In [ ]:
y_val = letters_new.iloc[12000:16000,16]
y_val.head()


12000    24.0
12001     2.0
12002     0.0
12003     1.0
12004    12.0
Name: y, dtype: float64

Fit a new scaler on the reduced training split.

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.astype(np.float64))


Retrain Random Forest, MLP, and SVM on the 12k training subset.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rnd_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rnd_clf.fit(X_train_scaled, y_train)


,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [ ]:
from sklearn.model_selection import cross_val_score
cross_val_score(rnd_clf, X_train_scaled, y_train, cv=3, scoring="accuracy")


array([0.94325, 0.9445 , 0.94   ])

Collect Random Forest predictions on the validation set.

In [ ]:
X_val_scaled = scaler.transform(X_val.astype(np.float64))
y_val_pred = rnd_clf.predict(X_val_scaled)


In [ ]:
from sklearn.neural_network import MLPClassifier
mlp_clf = MLPClassifier(random_state=42)
mlp_clf.fit(X_train_scaled, y_train)


/Users/mikaeldaluz/Documents/CSE4502/code/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,hidden_layer_sizes,"(100,)"
,activation,'relu'
,solver,'adam'
,alpha,0.0001
,batch_size,'auto'
,learning_rate,'constant'
,learning_rate_init,0.001
,power_t,0.5
,max_iter,200
,shuffle,True
,random_state,42


In [ ]:
cross_val_score(mlp_clf, X_train_scaled, y_train, cv=3, scoring="accuracy")


/Users/mikaeldaluz/Documents/CSE4502/code/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/mikaeldaluz/Documents/CSE4502/code/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/mikaeldaluz/Documents/CSE4502/code/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


array([0.92675, 0.9325 , 0.92075])

Collect MLP predictions on the validation set.

In [ ]:
y_val_pred_mlp = mlp_clf.predict(X_val_scaled)


In [ ]:
from sklearn.svm import SVC
from sklearn.multiclass import OneVsRestClassifier
ovr_clf = OneVsRestClassifier(SVC(gamma="auto", random_state=42, probability=True))
ovr_clf.fit(X_train_scaled, y_train)


,estimator,SVC(gamma='au...ndom_state=42)
,n_jobs,None
,verbose,0
,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'auto'
,coef0,0.0
,shrinking,True
,probability,True
,tol,0.001


In [ ]:
cross_val_score(ovr_clf, X_train_scaled, y_train, cv=3, scoring="accuracy")


array([0.9015 , 0.916  , 0.89075])

Collect SVM predictions on the validation set.

In [ ]:
y_val_pred_ovr = ovr_clf.predict(X_val_scaled)


Concatenate the three base-model prediction vectors into a meta-feature matrix.

In [ ]:
X_stack_pred_training=np.c_[y_val_pred, y_val_pred_mlp, y_val_pred_ovr]
print(X_stack_pred_training.shape)


(4000, 3)


In [ ]:
print(X_stack_pred_training)


[[24. 24. 24.]
 [ 2.  2.  2.]
 [ 0.  0.  0.]
 ...
 [ 6.  6.  6.]
 [ 4.  4. 25.]
 [ 2.  2.  2.]]


**Meta-learner (attempt 1):** Train a random forest blender on raw stacked predictions.

In [ ]:
blending_rnd_clf = RandomForestClassifier(n_estimators=100, random_state=42)
blending_rnd_clf.fit(X_stack_pred_training, y_val)


,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [ ]:
from sklearn.model_selection import cross_val_score
cross_val_score(blending_rnd_clf, X_stack_pred_training, y_val, cv=3, scoring="accuracy")


array([0.95502249, 0.94673668, 0.95123781])

**Meta-learner (attempt 2):** Train an MLP blender on raw stacked predictions — treating class indices as continuous values hurts performance.

In [ ]:
blending_mlp_clf = MLPClassifier(random_state=42)
blending_mlp_clf.fit(X_stack_pred_training, y_val)
cross_val_score(blending_mlp_clf, X_stack_pred_training, y_val, cv=3, scoring="accuracy")


/Users/mikaeldaluz/Documents/CSE4502/code/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/mikaeldaluz/Documents/CSE4502/code/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/mikaeldaluz/Documents/CSE4502/code/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/mikaeldaluz/Documents/CSE4502/code/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached 

array([0.63718141, 0.65341335, 0.6324081 ])

**Fix:** One-hot encode stacked predictions so the meta-learner treats them as categorical inputs rather than ordinal numbers.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
onehot_encoder = OneHotEncoder()
X_stack_pred_training_1hot = onehot_encoder.fit_transform(X_stack_pred_training)


In [ ]:
X_stack_pred_training_1hot.shape


(4000, 78)

In [ ]:
blending_rnd_clf_1hot = RandomForestClassifier(n_estimators=100, random_state=42)
blending_rnd_clf_1hot.fit(X_stack_pred_training_1hot, y_val)
cross_val_score(blending_rnd_clf_1hot, X_stack_pred_training_1hot, y_val, cv=3, scoring="accuracy")


array([0.96476762, 0.95048762, 0.96324081])

One-hot encoding improves the random forest blender's cross-validation accuracy.

In [ ]:
blending_mlp_clf_1hot = MLPClassifier(random_state=42)
blending_mlp_clf_1hot.fit(X_stack_pred_training_1hot, y_val)
cross_val_score(blending_mlp_clf_1hot, X_stack_pred_training_1hot, y_val, cv=3, scoring="accuracy")


/Users/mikaeldaluz/Documents/CSE4502/code/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/mikaeldaluz/Documents/CSE4502/code/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/mikaeldaluz/Documents/CSE4502/code/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


array([0.95952024, 0.94823706, 0.96474119])

The one-hot MLP blender also improves significantly. Random forest blender performs best overall, so it is used for final test evaluation.

In [ ]:
X_test_scaled = scaler.transform(X_test.astype(np.float64))
y_test_pred_rnd = rnd_clf.predict(X_test_scaled)
y_test_pred_mlp = mlp_clf.predict(X_test_scaled)
y_test_pred_ovr = ovr_clf.predict(X_test_scaled)
X_stack_pred_test = np.c_[y_test_pred_rnd,y_test_pred_mlp,y_test_pred_ovr]
X_stack_pred_test_1hot = onehot_encoder.transform(X_stack_pred_test)
y_test_pred_blending = blending_rnd_clf_1hot.predict(X_stack_pred_test_1hot)
print(blending_rnd_clf_1hot.__class__.__name__, accuracy_score(y_test, y_test_pred_blending))


RandomForestClassifier 0.95425


## 8. Final Results

The stacking classifier outperforms each individual model and hard voting. Soft voting achieves the highest accuracy among all methods tested.